# PASD support groups → spreadsheet

**What this does:** visits the PASD "Search for Support Groups" web page, copies the info for every group, sorts it into neat columns (address, ZIP, day, time, how often…), finds each place on a map, and gives you a spreadsheet (CSV).

**You don't need to know any code.** The code is hidden. Just:

1. Click **Runtime → Run all** in the menu at the top.
2. Wait about 1 minute. Each step shows a ✅ when it's done.
3. A file called `pasd_support_groups_scraped.csv` downloads at the end. Open it in Excel or Google Sheets.

If your browser asks to allow downloads, click **Allow**.

In [ ]:
# @title Step 1: Get set up (loads the tools)
# @markdown Installs a mapping tool and loads the instructions the later steps use. Nothing to change here.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "geopy"], check=True)
import re, requests
import pandas as pd

import re
from bs4 import BeautifulSoup

def _txt(el):
    if el is None:
        return ""
    t = re.sub(r"\s+", " ", el.get_text(" ").replace("\xa0", " ")).strip()
    return re.sub(r"\s+,", ",", t)

def _block_text(el):
    """Text of an HTML block, one line per paragraph/div/<br>, so line-based rules work."""
    if el is None:
        return ""
    el = BeautifulSoup(str(el), "html.parser")          # work on a copy
    for br in el.find_all("br"):
        br.replace_with("\n")
    for b in el.find_all(["p", "div", "h1", "h2", "h3", "h4", "h5", "h6", "li"]):
        b.append("\n")
    text = el.get_text("").replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    return "\n".join(l.strip() for l in text.split("\n") if l.strip())

def scrape_cards(html):
    soup = BeautifulSoup(html, "html.parser")
    cards, seen = [], set()
    for c in soup.select(".ex-tplist .item-grid"):
        post_id = c.get("data-id", "").split("-")[-1]
        if post_id in seen:
            continue
        seen.add(post_id)
        meta = c.select(".exp-lightbox-meta h5")
        def meta_val(label):
            h = next((h for h in meta if _txt(h).startswith(label)), None)
            return _txt(h)[len(label):].strip(" :") if h else ""
        cards.append({
            "name": _txt(c.select_one("figcaption h3")) or _txt(c.select_one(".lb-info h3")),
            "summary": _txt(c.select_one("figcaption p")),
            "cats": meta_val("Categories"),
            "locs": meta_val("Location"),
            "group": meta_val("Group"),
            "meta_phone": meta_val("Phone"),
            "meta_email": meta_val("Email"),
            "meta_contact": meta_val("Contact"),
            "desc": _block_text(c.select_one(".extp-mb-content")),
        })
    return cards

import re

DAYS = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
ORD_WORDS = {"first":"1st","second":"2nd","third":"3rd","fourth":"4th","fifth":"5th","last":"last",
             "1st":"1st","2nd":"2nd","3rd":"3rd","4th":"4th","5th":"5th"}
ORD = r"(?:1st|2nd|3rd|4th|5th|first|second|third|fourth|fifth|last)"
# e.g. "4th Friday", "First, second & third Tuesdays", "1st Thursday / Primer Jueves"
SCHEDULE_RE = re.compile(
    rf"\b({ORD}(?:\s*(?:,|&|and)\s*(?:and\s+)?{ORD})*)\s+(monday|tuesday|wednesday|thursday|friday|saturday|sunday)s?\b", re.I)

# Street address: house number + street name + street type, e.g. "1636 E. Mission Rd"
STREET_TYPES = r"(?:Street|St|Avenue|Ave|Road|Rd|Drive|Dr|Way|Parkway|Pkwy|Real|Boulevard|Blvd|Lane|Ln|Court|Ct|Place|Pl)"
ADDRESS_RE = re.compile(rf"\b(\d{{2,6}}\s+(?:[NSEW]\.?\s+)?[A-Za-z0-9.'’ ]+?\s{STREET_TYPES}\b\.?(?:,?\s*Suite\s*\w+)?)")
ZIP_RE = re.compile(r"\b(9[0-6]\d{3})\b")
CITIES = ["Chula Vista","La Mesa","Fallbrook","Carlsbad","Oceanside","Rancho Bernardo","Santee","Temecula","Poway",
          "Escondido","El Cajon","Vista","San Marcos","Encinitas","National City","Coronado","Lakeside","Ramona",
          "Alpine","Murrieta","Del Mar","Solana Beach","Imperial Beach","Spring Valley","San Diego"]
CITY_REGION = {"Chula Vista":"South Bay","National City":"South Bay","Imperial Beach":"South Bay","Coronado":"South Bay",
               "La Mesa":"East County","Santee":"East County","El Cajon":"East County","Lakeside":"East County",
               "Alpine":"East County","Ramona":"East County","Spring Valley":"East County",
               "Fallbrook":"North County","Carlsbad":"North County","Oceanside":"North County","Rancho Bernardo":"North County",
               "Escondido":"North County","Vista":"North County","San Marcos":"North County","Encinitas":"North County",
               "Poway":"North County","Del Mar":"North County","Solana Beach":"North County",
               "Temecula":"Riverside County","Murrieta":"Riverside County","San Diego":"Central County"}

# Times like "1-3pm", "10 am – Noon", "6:00 pm- 7:30", "3:30 – 4:30 PM", or a single "11am"
T = r"\d{1,2}(?::\d{2})?"
RANGE_RE = re.compile(rf"\b({T})\s*(am|pm)?\s*(?:-|–|to)\s*({T}|noon)\s*(am|pm)?", re.I)
SINGLE_RE = re.compile(rf"\b({T})\s*(am|pm)\b", re.I)
PHONE_RE = re.compile(r"\(?\b\d{3}\)?[\s.-]?\d{3}[\s.-]\d{4}\b")
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
NAME_NEAR_CONTACT_RE = re.compile(r"(?:Contact:? +|contact +)([A-Z][a-z]+(?: [A-Z][a-z]+)+)")
NAME_BEFORE_PHONE_RE = re.compile(r"([A-Z][a-z]+ [A-Z][a-z]+):? +\(?\d{3}\)?[ .-]?\d{3}")

def to_24h(t, ampm):
    if t.lower() == "noon":
        return "12:00"
    h, _, m = t.partition(":")
    h, m = int(h), int(m or 0)
    if ampm and ampm.lower() == "pm" and h != 12: h += 12
    if ampm and ampm.lower() == "am" and h == 12: h = 0
    return f"{h:02d}:{m:02d}"

def parse_time(text):
    """Return start-end in 24h time, e.g. '13:00-15:00', or '' if no time found."""
    for m in RANGE_RE.finditer(text):
        s, sap, e, eap = m.groups()
        if not (sap or eap or e.lower() == "noon"):
            continue                      # no am/pm anywhere: probably not a time
        if e.lower() == "noon": eap = "pm"
        sap = sap or eap                  # "1-3pm" -> start is pm too
        eap = eap or sap                  # "6:00 pm- 7:30" -> end is pm too
        start, end = to_24h(s, sap), to_24h(e, eap)
        if start > end and sap == eap:    # "11-1pm" -> start was am
            start = to_24h(s, "am")
        return f"{start}-{end}"
    m = SINGLE_RE.search(text)
    return to_24h(m.group(1), m.group(2)) if m else ""

def parse_schedule(text):
    """Find '<ordinal(s)> <weekday>' and turn it into day, week(s) and a clean frequency."""
    m = SCHEDULE_RE.search(text)
    low = text.lower()
    if not m:
        freq = "Quarterly" if "quarterly" in low else "Weekly" if re.search(r"\bweekly\b|every week", low) \
               else "Monthly" if "monthly" in low else "Unknown"
        return {"DAY_OF_WEEK": "", "WEEK_OF_MONTH": "", "FREQUENCY_CLEAN": freq, "FREQUENCY_RAW_MATCH": ""}
    ords = [ORD_WORDS[o.lower()] for o in re.findall(ORD, m.group(1), re.I)]
    n = len(ords)
    if "quarterly" in low: freq = "Quarterly"
    elif n >= 4:           freq = "Weekly"
    elif n == 2:           freq = "Twice monthly"
    elif n == 3:           freq = "3x monthly"
    else:                  freq = "Monthly"
    return {"DAY_OF_WEEK": m.group(2).capitalize(), "WEEK_OF_MONTH": ", ".join(ords),
            "FREQUENCY_CLEAN": freq, "FREQUENCY_RAW_MATCH": m.group(0)}

def find_line(text, needle):
    for line in text.split("\n"):
        if needle and needle.lower() in line.lower():
            return line.strip()
    return ""

def parse_address(desc):
    """Street, city, ZIP and venue name from the free-text description."""
    m = ADDRESS_RE.search(desc)
    if not m:
        return {"STREET_ADDRESS": "", "CITY": "", "ZIP_CODE": "", "VENUE_NAME": ""}
    street = m.group(1).strip()
    after = desc[m.end(): m.end() + 60]
    window = street + " " + after.split("\n")[0] + " " + (after.split("\n")[1] if "\n" in after else "")
    city = next((c for c in CITIES if re.search(rf"\b{c}\b", window)), "")
    z = ZIP_RE.search(after[:40])
    # Venue: text before the address on the same line, else the previous line
    line_start = desc.rfind("\n", 0, m.start()) + 1
    before = desc[line_start:m.start()]
    before = re.split(r"Ubicación:|Location:|WHERE:", before)[-1].strip(" ,:-")
    if not before:
        prev_lines = [l for l in desc[:line_start].split("\n") if l.strip()]
        before = prev_lines[-1] if prev_lines else ""
        before = re.sub(r"^(WHERE|Location|Ubicación):\s*", "", before.strip(), flags=re.I)
    return {"STREET_ADDRESS": street, "CITY": city, "ZIP_CODE": z.group(1) if z else "", "VENUE_NAME": before}

def venue_type(venue, virtual):
    v = venue.lower()
    if virtual: return "Virtual"
    for words, label in [(("church","presbyterian","methodist","catholic","lutheran"), "Church"),
                         (("senior center",), "Senior center"), (("senior living",), "Senior living community"),
                         (("library",), "Library"), (("cafe","restaurant"), "Restaurant/cafe"), (("gym",), "Gym"),
                         (("health","wellness","hospital","clinic"), "Health center")]:
        if any(w in v for w in words): return label
    return "Other"

def audience(cats, text):
    c, t = cats.lower(), text.lower()
    if "care partners only" in c or "caregivers only" in t:
        a = "Care partners only"
    elif "pwp only" in c:
        a = "People with Parkinson's only"
    elif "both" in c:
        a = "PwP and care partners"
    elif re.search(r"for care partners", t):
        a = "Care partners"
    else:
        a = "Not stated"
    if "young onset" in c or "yopd" in t: a += " (young onset)"
    if "atypical" in c: a += " (atypical PD)"
    if "women" in t: a += " (women)"
    return a

def region(locs, city, virtual):
    if virtual: return "Virtual"
    tags = [l.strip() for l in locs.split(",") if l.strip() and "San Diego" not in l and l.strip() != "Virtual"]
    if len(tags) == 1: return tags[0]
    return CITY_REGION.get(city, "")   # site tags missing or too many: fall back to the city

def contacts(text):
    # Drop Zoom meeting IDs and dial-in lines so they aren't mistaken for phone numbers
    text = "\n".join(l for l in text.split("\n")
                     if not re.search(r"meeting id|passcode|zoom|^\s*\+|#|^\s*\d\. phone", l, re.I))
    phones = list(dict.fromkeys(PHONE_RE.findall(text)))
    emails = list(dict.fromkeys(EMAIL_RE.findall(text)))
    m = NAME_NEAR_CONTACT_RE.search(text) or NAME_BEFORE_PHONE_RE.search(text)
    return {"CONTACT_NAME": m.group(1) if m else "", "CONTACT_PHONE": "; ".join(phones[:2]),
            "CONTACT_EMAIL": "; ".join(e.rstrip(".") for e in emails[:2])}

def clean_card(card):
    """Turn one scraped card (dict of raw text) into one dataset row."""
    desc, summary = card["desc"], card["summary"]
    both = summary + "\n" + desc
    addr = parse_address(desc)
    virtual = not addr["STREET_ADDRESS"] and bool(re.search(r"zoom|virtual", both + card["locs"], re.I))
    sched = parse_schedule(desc)                 # prefer the full description...
    sched_src = desc
    if not sched["DAY_OF_WEEK"]:                 # ...fall back to the short summary line
        sched, sched_src = parse_schedule(both), both
    row = {
        "EVENT_NAME": card["name"],
        "GROUP_ORG": card["group"],
        "EVENT_TYPE": venue_type(addr["VENUE_NAME"], virtual),
        "VENUE_NAME": "Zoom" if virtual else addr["VENUE_NAME"],
        "STREET_ADDRESS": "virtual" if virtual else addr["STREET_ADDRESS"],
        "ZIP_CODE": addr["ZIP_CODE"], "CITY": addr["CITY"], "STATE": "" if virtual else "CA",
        "FREQUENCY_RAW": find_line(sched_src, sched["FREQUENCY_RAW_MATCH"]) or summary,
        "FREQUENCY_CLEAN": sched["FREQUENCY_CLEAN"],
        "DAY_OF_WEEK": sched["DAY_OF_WEEK"], "WEEK_OF_MONTH": sched["WEEK_OF_MONTH"],
        "TIME_LOCAL": parse_time(desc) or parse_time(summary),
        "FORMAT": "Virtual" if virtual else "In-person",
        "AUDIENCE": audience(card["cats"], both),
        "LANGUAGE": "Spanish" if re.search(r"español|compañeros", both, re.I) else "English",
        "REGION": region(card["locs"], addr["CITY"], virtual),
        "SITE_CATEGORIES": card["cats"], "SITE_LOCATIONS": card["locs"],
        **contacts(desc),
    }
    # Fall back to the labelled fields at the top of the card (Phone:, Email:, Contact:)
    for col, key in [("CONTACT_NAME", "meta_contact"), ("CONTACT_PHONE", "meta_phone"), ("CONTACT_EMAIL", "meta_email")]:
        if not row[col] and card.get(key):
            row[col] = card[key]
    flags = []
    if not virtual and not row["ZIP_CODE"]: flags.append("ZIP missing on site")
    if not virtual and not row["CITY"]: flags.append("city missing on site")
    if not row["DAY_OF_WEEK"]: flags.append("day not listed")
    if not row["TIME_LOCAL"]: flags.append("time not listed")
    until = re.search(r"until ([A-Z][a-z]{2} \d{1,2}, \d{4})", desc)
    if until: flags.append(f"listing says 'until {until.group(1)}', may be outdated")
    row["NOTES"] = "; ".join(flags)
    return row

print("✅ Set up done")

In [ ]:
# @title Step 2: Copy the support groups from the PASD website
# @markdown Downloads the web page and pulls out each group's card (name, description, contacts).
URL = "https://parkinsonsassociation.org/search-for-support-groups/"
try:
    resp = requests.get(URL, headers={"User-Agent": "Mozilla/5.0 (Biokind UCLA student project)"}, timeout=30)
    resp.raise_for_status()
    html = resp.text
except Exception as e:
    print("⚠️ The website didn't let us download the page.")
    print("Fix: open the page in Chrome, press Ctrl+S (Cmd+S on Mac) and save it as support_groups.html,")
    print("then drag that file into the folder icon on the left side of Colab and click Runtime → Run all again.")
    html = open("support_groups.html", encoding="utf-8").read()

cards = scrape_cards(html)
print(f"✅ Found {len(cards)} support groups")

In [ ]:
# @title Step 3: Sort the text into columns
# @markdown Reads each description and finds the address, ZIP, day ("4th Friday"), time ("1-3pm"), how often it meets, and whether it's on Zoom. Also adds Chula Vista Central, which isn't on the website yet.
df = pd.DataFrame([clean_card(c) for c in cards])

new_group = {
    "EVENT_NAME": "Chula Vista Central", "EVENT_TYPE": "Church", "VENUE_NAME": "First United Methodist Church",
    "STREET_ADDRESS": "1200 E H Street", "ZIP_CODE": "91910", "CITY": "Chula Vista", "STATE": "CA",
    "FREQUENCY_CLEAN": "TBD", "FORMAT": "In-person", "AUDIENCE": "TBD", "LANGUAGE": "English",
    "REGION": "South Bay", "CONTACT_NAME": "Anna Silsbee",
    "NOTES": "New group from project outline, not on website yet; schedule needed from PASD"}
df = pd.concat([df, pd.DataFrame([new_group])], ignore_index=True).fillna("")
df["SOURCE"] = [URL] * (len(df) - 1) + ["Project outline"]
print(f"✅ {len(df)} groups sorted into columns")
df[["EVENT_NAME", "CITY", "DAY_OF_WEEK", "WEEK_OF_MONTH", "TIME_LOCAL", "FREQUENCY_CLEAN", "FORMAT"]]

In [ ]:
# @title Step 4: Find each place on the map (takes ~25 seconds)
# @markdown Looks up each in-person address on OpenStreetMap to get map coordinates. If the website left out a ZIP or city, this fills it in and says so in the NOTES column.
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
geocode = RateLimiter(Nominatim(user_agent="biokind-pasd-support-group-map").geocode, min_delay_seconds=1.1)

def add_note(i, note):
    df.at[i, "NOTES"] = "; ".join(x for x in [df.at[i, "NOTES"], note] if x)

df["LAT"], df["LON"] = "", ""
for i, r in df[df["FORMAT"] != "Virtual"].iterrows():
    street = re.sub(r",?\s*Suite\s*\w+", "", r["STREET_ADDRESS"])
    area = r["CITY"] or "San Diego County"
    loc = None
    for q in [", ".join(x for x in [street, area, "CA", r["ZIP_CODE"]] if x), f"{street}, {area}, CA", f"{r['VENUE_NAME']}, {area}, CA"]:
        loc = geocode(q, addressdetails=True, country_codes="us")
        if loc:
            break
    if not loc:
        add_note(i, "could not find on map")
        continue
    df.at[i, "LAT"], df.at[i, "LON"] = f"{loc.latitude:.5f}", f"{loc.longitude:.5f}"
    a = loc.raw.get("address", {})
    if not r["ZIP_CODE"] and a.get("postcode"):
        df.at[i, "ZIP_CODE"] = a["postcode"][:5]; add_note(i, "ZIP filled from map lookup")
    if not r["CITY"]:
        df.at[i, "CITY"] = a.get("city") or a.get("town") or a.get("village") or ""; add_note(i, "city filled from map lookup")
    if not df.at[i, "REGION"]:
        df.at[i, "REGION"] = CITY_REGION.get(df.at[i, "CITY"], "")
print("✅ Map coordinates added")

In [ ]:
# @title Step 5: Save and download the spreadsheet
COLUMNS = ["EVENT_NAME", "GROUP_ORG", "EVENT_TYPE", "VENUE_NAME", "STREET_ADDRESS", "ZIP_CODE", "CITY", "STATE",
           "FREQUENCY_RAW", "FREQUENCY_CLEAN", "DAY_OF_WEEK", "WEEK_OF_MONTH", "TIME_LOCAL", "FORMAT", "AUDIENCE",
           "LANGUAGE", "REGION", "CONTACT_NAME", "CONTACT_PHONE", "CONTACT_EMAIL", "LAT", "LON",
           "SITE_CATEGORIES", "SITE_LOCATIONS", "SOURCE", "NOTES"]
df[COLUMNS].sort_values("EVENT_NAME").to_csv("pasd_support_groups_scraped.csv", index=False)
print("✅ Saved pasd_support_groups_scraped.csv. Check the NOTES column for anything to double-check.")
try:
    from google.colab import files
    files.download("pasd_support_groups_scraped.csv")
except ImportError:
    pass

### How to describe this in your write-up

*"I wrote a Python notebook that scrapes the PASD support groups page (requests + BeautifulSoup), uses regular expressions to standardize the address, ZIP, meeting day, time and frequency, adds the new Chula Vista Central group, geocodes in-person addresses with OpenStreetMap, and exports a CSV. Values the code had to infer are flagged in a NOTES column."*

**Want to see the code?** Click **Show code** on any step.